In [6]:
import sys, os, tempfile, timeit, pickle, inspect
from pathlib import Path
from dsc.dsc_io import load_dsc as __load_dsc__, source_dirs as __source_dirs__
import numpy as np
sys.path.append("/gpfs/commons/home/sbanerjee/work/npd/lrma-dsc/dsc/functions")
from comparison_metrics import (
    global_calibration,
    rebalance_to_unit_F,
    coupled_procrustes_per_factor_scaling,
    root_mean_squared_error,
    relative_rmse,
    peak_signal_to_noise_ratio,
    adjusted_mutual_information_score
)

In [12]:
DSC_RESDIR = f'/gpfs/commons/groups/knowles_lab/sbanerjee/' + (
        f'low_rank_matrix_approximation_numerical_experiments/lrma')
DSC_C1BF078C = dict()
DSC_C1BF078C = __load_dsc__(
    [os.path.join(DSC_RESDIR, 'blockdiag_k/blockdiag_k_1.pkl'),
     os.path.join(DSC_RESDIR, 'truncated_svd/blockdiag_k_1_nnm_1_truncated_svd_1.pkl')])
DSC_REPLICATE = DSC_C1BF078C["DSC_DEBUG"]["replicate"]
DSC_SEED = DSC_C1BF078C["DSC_DEBUG"]["seed"] + 27
F = DSC_C1BF078C['F_est']
Ftrue = DSC_C1BF078C['Ftrue']
L = DSC_C1BF078C['L_est']
Ltrue = DSC_C1BF078C['Ltrue']
labels = DSC_C1BF078C['Ctrue']
for name, func in __source_dirs__(['functions']):
    globals()[name] = func
TIC_C1BF078C = timeit.default_timer()
DSC_SEED += DSC_REPLICATE
import random
random.seed(DSC_SEED)
try:
	import numpy
	numpy.random.seed(DSC_SEED)
except Exception:
	pass

In [13]:
Ltrue.shape

(200, 2)

In [66]:
def coupled_procrustes_per_factor_scaling(L_true, F_true, L_hat, F_hat, dim_policy="zerofill", eps=1e-8):
    
    from comparison_metrics import match_latent_dimensions
    from scipy.linalg import orthogonal_procrustes
    
    def has_no_matrix_signal(X):
        #return (np.linalg.norm(X, ord="fro") > eps) and (np.ptp(X) > eps)
        return np.ptp(X) == 0

    F_true_m, F_hat_m = match_latent_dimensions(F_true, F_hat, dim_policy = dim_policy)
    L_true_m, L_hat_m = match_latent_dimensions(L_true, L_hat, dim_policy = dim_policy)

    # gleanr sometimes produces a single column of zero values
    # We can't rescue with alignment. In fact, procrustes returns error.
    if has_no_matrix_signal(F_hat_m):
        k = F_hat_m.shape[1]
        R = np.eye(k) # no rotation, identity matrix
        scales_safe = np.ones(k, dtype=float)
        F_aligned = F_hat_m.copy()
        L_aligned = L_hat_m.copy()
    else:
        # Shared orthogonal alignment
        R, _ = orthogonal_procrustes(F_hat_m, F_true_m)
    
        F_rot = F_hat_m @ R
        L_rot = L_hat_m @ R
    
        # One scale per aligned column
        numer = np.sum(F_rot * F_true_m, axis=0)
        denom = np.sum(F_rot * F_rot, axis=0)
        scales = np.ones(F_rot.shape[1], dtype=float)

        # Columns that are truly absent in the simulation truth.
        # These are padded background columns when K_true < K_hat.
        true_zero_cols = (
            np.sum(F_true_m * F_true_m, axis=0) <= eps**2
        ) & (
            np.sum(L_true_m * L_true_m, axis=0) <= eps**2
        )

        # mask for columns with enough signal and true non-zero
        fit_mask = (denom > eps**2) & (~true_zero_cols)
        # fit_mask = denom > eps**2
        scales[fit_mask] = numer[fit_mask] / denom[fit_mask]

        # Clip nearly zero scales for numerical stability. But, keep the sign.
        # Only protect genuinely fitted scales, not padded/invalid columns
        tiny_valid = fit_mask & (np.abs(scales) < eps)
        scales_safe = scales.copy()
        # scales_safe[tiny_valid] = np.where(scales_safe[tiny_valid] >= 0, eps, -eps)
        # For true-zero columns, just scale with F_norm
        # This penalizes extra estimated factors directly instead of exploding L.
        F_rot_norm = np.sqrt(np.sum(F_rot * F_rot, axis=0))
        scaleable_true_zero_cols = true_zero_cols & (F_rot_norm > eps)
        scales_safe[scaleable_true_zero_cols] = 1.0 / F_rot_norm[scaleable_true_zero_cols]

    
        F_aligned = F_rot * scales_safe.reshape(1, -1)
        L_aligned = L_rot / scales_safe.reshape(1, -1)

    return {
        "L_true": L_true_m,
        "F_true": F_true_m,
        "L_aligned": L_aligned,
        "F_aligned": F_aligned,
        "rotation": R,
        "scales": scales_safe,
    }

In [14]:
# True signal
Zt = Ltrue @ Ftrue.T

# Raw estimated signal
Z = L @ F.T

# Oracle alignment on raw data
aligned_raw = coupled_procrustes_per_factor_scaling(Ltrue, Ftrue, L, F, dim_policy="zerofill")
Lt_raw = aligned_raw["L_true"]
Ft_raw = aligned_raw["F_true"]
La_raw = aligned_raw["L_aligned"]
Fa_raw = aligned_raw["F_aligned"]
scales_raw = aligned_raw["scales"]

# our simulations use orthonormal factors.
# Rebalance L and F enforcing ||F||_2 = 1
L_bal, F_bal, balancing_impact = rebalance_to_unit_F(L, F)

# Oracle alignment after balancing
aligned_bal = coupled_procrustes_per_factor_scaling(Ltrue, Ftrue, L_bal, F_bal, dim_policy="zerofill")
Lt_bal = aligned_bal["L_true"]
Ft_bal = aligned_bal["F_true"]
La_bal = aligned_bal["L_aligned"]
Fa_bal = aligned_bal["F_aligned"]
scales_bal = aligned_bal["scales"]

# some methods introduce attenuation bias
# remove only the attenuation bias while preserving row-wise and column-wise structure.
# At zero norm, we force L and F to zero.
sg, is_zero_norm = global_calibration(Zt, Z)
Za = sg * Z
# Assign global calibration to L by convention
L_cal = sg * L
F_cal = np.zeros_like(F) if is_zero_norm else F.copy()
L_bal_cal = sg * L_bal
F_bal_cal = np.zeros_like(F_bal) if is_zero_norm else F_bal.copy()

# Oracle alignment after global_calibration
aligned_cal = coupled_procrustes_per_factor_scaling(Ltrue, Ftrue, L_cal, F_cal, dim_policy="zerofill")
Lt_cal = aligned_cal["L_true"]
Ft_cal = aligned_cal["F_true"]
La_cal = aligned_cal["L_aligned"]
Fa_cal = aligned_cal["F_aligned"]
scales_cal = aligned_cal["scales"]

# Oracle alignment after balancing + global calibration
aligned_bal_cal = coupled_procrustes_per_factor_scaling(Ltrue, Ftrue, L_bal_cal, F_bal_cal, dim_policy="zerofill")
Lt = aligned_bal_cal["L_true"]
Ft = aligned_bal_cal["F_true"]
La = aligned_bal_cal["L_aligned"]
Fa = aligned_bal_cal["F_aligned"]
scales = aligned_bal_cal["scales"]

# RMSE
# align
L_rmse_raw = root_mean_squared_error(Lt_raw, La_raw)
F_rmse_raw = root_mean_squared_error(Ft_raw, Fa_raw)
Z_rmse_raw = root_mean_squared_error(Zt, Z)
# balance + align
L_rmse_bal = root_mean_squared_error(Lt_bal, La_bal)
F_rmse_bal = root_mean_squared_error(Ft_bal, Fa_bal)
Z_rmse_bal = root_mean_squared_error(Zt, Z) # balancing does not change Z.
# calibrate + align
L_rmse_cal = root_mean_squared_error(Lt_cal, La_cal)
F_rmse_cal = root_mean_squared_error(Ft_cal, Fa_cal)
Z_rmse_cal = root_mean_squared_error(Zt, Za)
# balance + calibrate + align
L_rmse = root_mean_squared_error(Lt, La)
F_rmse = root_mean_squared_error(Ft, Fa)
Z_rmse = root_mean_squared_error(Zt, Za)

# Relative RMSE / relative Frobenius error
# align
L_rel_rmse_raw = relative_rmse(L_rmse_raw, Lt)
F_rel_rmse_raw = relative_rmse(F_rmse_raw, Ft)
Z_rel_rmse_raw = relative_rmse(Z_rmse_raw, Zt)
# balance + align
L_rel_rmse_bal = relative_rmse(L_rmse_bal, Lt)
F_rel_rmse_bal = relative_rmse(F_rmse_bal, Ft)
Z_rel_rmse_bal = relative_rmse(Z_rmse_bal, Zt)
# calibrate + align
L_rel_rmse_cal = relative_rmse(L_rmse_cal, Lt)
F_rel_rmse_cal = relative_rmse(F_rmse_cal, Ft)
Z_rel_rmse_cal = relative_rmse(Z_rmse_cal, Zt)
# balance + calibrate + align
L_rel_rmse = relative_rmse(L_rmse, Lt)
F_rel_rmse = relative_rmse(F_rmse, Ft)
Z_rel_rmse = relative_rmse(Z_rmse, Zt)

# Optional PSNR; not primary
L_psnr = peak_signal_to_noise_ratio(Lt, La)
F_psnr = peak_signal_to_noise_ratio(Ft, Fa)
Z_psnr = peak_signal_to_noise_ratio(Zt, Za)

# Clustering metrics
MI_raw, adj_MI_raw = adjusted_mutual_information_score(L, labels)
MI_bal, adj_MI_bal = adjusted_mutual_information_score(L_bal, labels)
MI_cal, adj_MI_cal = adjusted_mutual_information_score(L_cal, labels)
MI_bal_cal, adj_MI_bal_cal = adjusted_mutual_information_score(L_bal_cal, labels)
MI_oracle_aligned, adj_MI_oracle_aligned = adjusted_mutual_information_score(La, labels)

In [15]:
F_rel_rmse_cal

2.006239930947296

In [16]:
F_norm = np.sqrt(np.sum(F * F, axis=0))
Fa_raw_norm = np.sqrt(np.sum(Fa_raw * Fa_raw, axis=0))
Fa_bal_norm = np.sqrt(np.sum(Fa_bal * Fa_bal, axis=0))
Fa_cal_norm = np.sqrt(np.sum(Fa_cal * Fa_cal, axis=0))
Fa_norm = np.sqrt(np.sum(Fa * Fa, axis=0))

print ("F_norm =", F_norm)
print ("Fa_raw_norm =", Fa_raw_norm)
print ("Fa_bal_norm =", Fa_bal_norm)
print ("Fa_cal_norm =", Fa_cal_norm)
print ("Fa_norm =", Fa_norm)

F_norm = [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Fa_raw_norm = [0.9864883  0.98835394 1.         1.         1.         1.
 1.         1.         1.         1.        ]
Fa_bal_norm = [0.9864883  0.98835394 1.         1.         1.         1.
 1.         1.         1.         1.        ]
Fa_cal_norm = [0.9864883  0.98835394 1.         1.         1.         1.
 1.         1.         1.         1.        ]
Fa_norm = [0.9864883  0.98835394 1.         1.         1.         1.
 1.         1.         1.         1.        ]


In [69]:
F_diff = Fa_cal - Ft_cal
np.sqrt(np.sum(F_diff * F_diff)) / np.sqrt(8)

1.003119965473648

In [65]:
F_diff = Fa_cal[:, 2:] - Ft_cal[:, 2:]
np.sqrt(np.sum(F_diff * F_diff))

2.82842712474619

In [78]:
col_weights = np.sum(La_cal * La_cal, axis=0) / np.sum(La_cal*La_cal)
col_weights > 0.001

array([ True,  True, False, False, False, False, False, False, False,
       False])

In [50]:
total_norm = np.linalg.norm(La_cal, ord='fro')
excess_norm = np.linalg.norm(La_cal[:, 2:], ord='fro')
print (f"total norm = {total_norm:g}")
print (f"excess norm = {excess_norm:g}")

total norm = 3.56115
excess norm = 0.267412


In [52]:
total_norm = np.linalg.norm(La_cal - Lt_cal, ord='fro')
inner_norm = np.linalg.norm(La_cal[:, :2] - Lt_cal[:, :2], ord ='fro')
excess_norm = np.linalg.norm(La_cal[:, 2:] - Lt_cal[:, 2:], ord='fro')
print (f"total norm = {total_norm:g}")
print (f"inner norm = {inner_norm:g}")
print (f"excess norm = {excess_norm:g}")

total norm = 1.2282
inner norm = 1.19873
excess norm = 0.267412


In [57]:
np.linalg.norm(La_cal[:,:], ord='fro')

3.561151078683814

In [49]:
np.linalg.norm(La_cal[:, :2] - Ltrue, ord='fro')

1.1987322211498292

In [ ]:
Ft

In [64]:
La_raw[:5, 5:8] * sg

array([[-0.19976454,  0.17922272,  0.14119523],
       [-0.11293368, -0.10054357, -0.05119987],
       [ 0.0447423 , -0.02917023, -0.07230326],
       [ 0.00925139, -0.02182864, -0.01017795],
       [ 0.07685158, -0.04041197, -0.01331431]])

In [65]:
La_cal[:5, 5:8]

array([[-0.19976454,  0.17922272,  0.14119523],
       [-0.11293368, -0.10054357, -0.05119987],
       [ 0.0447423 , -0.02917023, -0.07230326],
       [ 0.00925139, -0.02182864, -0.01017795],
       [ 0.07685158, -0.04041197, -0.01331431]])

In [17]:
F_norm = np.sqrt(np.sum(F * F, axis=0))
L_norm = np.sqrt(np.sum(L * L, axis=0))

print("sg =", sg)
print("F_norm =", F_norm)
print("L_norm =", L_norm)
print("sg * F_norm =", sg * F_norm)
print("balancing_impact =", balancing_impact)

sg = 1794.7410024604017
F_norm = [0.96577296 0.98227634 0.96227183 0.97270375 0.95063575 0.02132007
 0.02132007 0.02132007 0.02132007 0.02132007]
L_norm = [ 0.84470305  0.83051106  0.84777641  0.8386843   0.85815347 38.26400674
 38.26400674 38.26400674 38.26400674 38.26400674]
sg * F_norm = [1733.31233216 1762.93161639 1727.02871512 1745.75130935 1706.14496123
   38.26400674   38.26400674   38.26400674   38.26400674   38.26400674]
balancing_impact = 1.6712113404111033


In [4]:
import glob
def check_L_est_dimension(outdir, file_ext):
    fnames = glob.glob(f"{outdir}/*.{file_ext}")
    dim_dict = dict()
    for f in fnames:
        fbase = os.path.basename(f)
        mres = __load_dsc__([f])
        dim_dict[fbase] = mres['L_est'].shape[1]
    return dim_dict

outdir = "/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/gleanr"
_mres_kdict = check_L_est_dimension(outdir, "rds")

In [5]:
_mres_kdict.values()

dict_values([3, 1, 5, 5, 3, 10, 5, 6, 2, 10, 7, 4, 10, 7, 10, 10, 4, 10, 2, 10, 10, 6, 4, 3, 10, 2, 2, 10, 10, 5, 10, 8, 4, 1, 9, 5, 10, 4, 2, 4, 10, 5, 10, 5, 5, 3, 10, 10, 10, 10, 10, 4, 6, 6, 5, 10, 10, 10, 4, 5, 10, 10, 6, 10, 3, 3, 1, 6, 8, 10, 6, 10, 10, 4, 2, 10, 2, 7, 10, 3, 10, 2, 10, 10, 5, 1, 10, 10, 5, 10, 10, 10, 3, 2, 2, 10, 1, 10, 10, 10, 1, 3, 4, 10, 10, 8, 4, 10, 10, 3, 10, 10, 4, 2, 5, 5, 8, 4, 4, 7, 2, 10, 10, 6, 2, 10, 6, 10, 10, 10, 10, 10, 10, 10, 5, 10, 4, 10, 2, 4, 10, 6, 6, 10, 5, 6, 10, 10, 10, 2, 6, 5, 4, 10, 5, 2, 6, 10, 2, 4, 6, 4, 10, 10, 4, 5, 10, 4, 6, 7, 10, 10, 2, 1, 9, 10, 5, 10, 3, 10, 3, 10, 10, 10, 5, 6, 10, 5, 6, 3])

In [46]:
_select_sims = [fname for fname, k in _mres_kdict.items()]
for f in _select_sims:
    mres = __load_dsc__([os.path.join(outdir, f)])
    if np.any(np.all(mres['L_est'] == mres['L_est'][0:1, :], axis = 0)):
        print(f)
        #print(mres['L_est'])

blockdiag_p_28_identical_1_gleanr_1.rds
blockdiag_p_32_identical_1_gleanr_1.rds


In [54]:
def _matrix_dissimilarity_scores(original, recovered, mask = None, match = 'zerofill'):
    n_orig = original.shape[1]
    n_recv = recovered.shape[1]
    if match == 'clip':
        n = min(n_orig, n_recv)
        X = original[:, :n]
        Y = recovered[:, :n]
    elif match == 'zerofill':
        m = original.shape[0]
        n = max(n_orig, n_recv)
        X = np.zeros((m, n))
        Y = np.zeros((m, n))
        X[:, :n_orig] = original
        Y[:, :n_recv] = recovered
    # gleanr sometimes produces a single column of zero values
    # procrustes requires: Input matrices must contain >1 unique points
    # a matrix has no unique points iff max - min == 0.
    if np.ptp(Y) == 0 :
        m2 = np.sum(np.square(X))
    else:
        R_orig, R_recv, m2 = procrustes(X, Y)
    psnr = peak_signal_to_noise_ratio(R_orig, R_recv, mask)
    return np.sqrt(m2), psnr

In [77]:
sys.path.append("/gpfs/commons/home/sbanerjee/work/npd/lrma-dsc/dsc/functions")

## BEGIN DSC CORE
import numpy as np
from comparison_metrics import matrix_dissimilarity_scores, adjusted_mutual_information_score, mean_squared_error, peak_signal_to_noise_ratio
L_rmse, L_psnr = matrix_dissimilarity_scores(Ltrue, L)
# F_rmse, F_psnr = matrix_dissimilarity_scores(Ftrue, F)
Ztrue = Ltrue @ Ftrue.T
Zrecv = L @ F.T
Ztrue = Ztrue - np.mean(Ztrue, axis = 0, keepdims = True)
Zrecv = Zrecv - np.mean(Zrecv, axis = 0, keepdims = True)
Z_rmse = np.sqrt(mean_squared_error(Ztrue, Zrecv))
Z_psnr = peak_signal_to_noise_ratio(Ztrue, Zrecv)
adj_MI = adjusted_mutual_information_score(L, labels)
## END DSC CORE

ValueError: Input matrices must contain >1 unique points

In [68]:
np.square(L_rmse) / 200

0.004981024159461685

In [75]:
np.sqrt(np.sum(np.square(Ltrue)))

4.100206872020673

In [69]:
mean_squared_error(Ltrue, L)

0.008405807375362704

In [48]:
Z_rmse

0.0026733336396964967

In [49]:
Z_psnr

22.249329802386598

In [50]:
adj_MI

-0.0020696076944895407